In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import os

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_path = "path/documents.txt"
index_path = "path/faiss.index"

def load_docs():
    with open(doc_path, "r") as f:
        return [line.strip() for line in f.readlines()]

def build_index():
    docs = load_docs()
    embeddings = model.encode(docs, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, index_path)
    return index, docs

if os.path.exists(index_path):
    index = faiss.read_index(index_path)
    documents = load_docs()
else:
    index, documents = build_index()

def retrieve_top_k(question, k=1):
    q_vec = model.encode([question], convert_to_numpy=True)
    D, I = index.search(q_vec, k)
    return [documents[i] for i in I[0]]
